### Structured Output
###### models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing . Langchain supposrts multiple schema types and methods for enforcing structured outputs.

### Pydantic
###### pydantic models provide the richest features set with validation , description and nested structures

In [ ]:
import os
from langchain.chat_models import init_chat_modesls

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_modesls(
    model="groq:llama-3.3-70b-versatile"
)
model

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=field(description="The title of movie")
    year:int=field(description="The year the movie was released")
    director:str=field(description="Name of the director of the movie")
    rating:float=field(description="The movie's rating out f 10")

In [ ]:
model_with_structured_output=model.with_structured_output(Movie)
response= model_with_structured_output.invoke("Provide details about the movie Inception")
response

### TypedDict
###### provides a simpler alternatice using python's built-in typing , idela when you dont need runtime validation.

In [ ]:
from typing_extensions import TypedDict, Annotations

class Movie(TypedDict):
    title:Annotate([str,...,"The title of the movie"])
    year: int
    cast: list[Actor]

model_with_structured_output= model.with_structured_output(Movie)
response=model_with_structured_output.invoke("")
print(response)

###### pydantic does runtime type coercion , whereas typedict believes completely in the llm response might or might not be correct
###### pydantic object vs python dictinoary
###### diff in how langchain process both are pconverted to json schema and sent to model 

#### DataClass
###### use datacalss decarator , it contains data and no restriction


In [ ]:
#create agent and integrate

from langchain.agents import create_agent
from pydantic import BaseModel, Field

class Contact(BaseModel):
    """contact information for a person."""
    name: str= Field(description="name of the person")
    phone: str= Field(description="contact number of the person")
    email: str=Field(description="email of the person")

agent = create_agent(
    model="",
    response_format=Contact
)

result = agent.invoke([
    "messages":[{"role":"user","content":"extract contact info from :jani jp jani@ex.com 9823-0123"}]
])
print(result["structured_response"])

In [ ]:
Why Pydantic?: It is the only option that provides runtime validation, automatic type coercion (e.g., fixing "5" to 5), and returns a rich object instance with methods.  This ensures the data is actually correct before it crashes your application.
When to use others:
TypedDict: Preferred for LangGraph state schemas where performance is critical and validation is handled elsewhere, or for very simple, internal dictionaries where overhead must be zero.
Dataclass: Rarely used for .with_structured_output() specifically, as it offers no validation advantage over TypedDict while returning a plain dict anyway. It is sometimes used for defining internal context or configuration objects.
